In [6]:
import arcpy
import os

# 1. 設定環境：指向目前的工程與預設資料庫 (Default.gdb)
aprx = arcpy.mp.ArcGISProject("CURRENT")
default_gdb = aprx.defaultGeodatabase
arcpy.env.workspace = default_gdb
arcpy.env.overwriteOutput = True # 允許覆寫同名檔案

# 2. 使用者參數設定
input_point_layer = "Random_Points_k1a2023_NC"  # 點圖層名稱
diameter_cm = 1                                   # 設定圓直徑 (1公分)
output_name = "k1a_random_1cm_Circle"           # 設定產出的 Polygon 名稱

# 3. 計算半徑
radius = diameter_cm / 2
buffer_distance = f"{radius} Centimeters"

# 4. 定義完整的輸出路徑
output_path = os.path.join(default_gdb, output_name)

try:
    print(f"正在處理... 使用平面投影計算半徑: {buffer_distance}")
    
    # 5. 執行環域分析 (Buffer)
    # 將 method 改為 "PLANAR"，在 GDB 中會產生 True Curves (完美圓形)
    arcpy.analysis.Buffer(
        in_features=input_point_layer,
        out_feature_class=output_path,
        buffer_distance_or_field=buffer_distance,
        line_side="FULL",
        line_end_type="ROUND",
        dissolve_option="NONE", 
        dissolve_field=[],
        method="PLANAR"       # 關鍵修改：使用平面法以支援真實曲線
    )

    print(f"成功！新的 Polygon 已儲存至: {output_path}")
    
except Exception as e:
    print(f"發生錯誤: {e}")

正在處理... 使用平面投影計算半徑: 0.5 Centimeters
成功！新的 Polygon 已儲存至: D:\ArcGIS\Morpho_Spatial_Demography\MSD_Hawaii.gdb\k1a_random_1cm_Circle


In [8]:
###圓形重疊判斷

import arcpy
import os

# 1. 設定輸入參數
circle_polygon = "k1a_random_1cm_Circle"  # 你的圓形圖層
target_polygon = "k1a_2023_crevice"          # 要判斷相疊的另一個圖層
field_name = "At_crevice"

try:
    # --- 前半段：空間判斷與屬性更新 ---
    
    # 檢查欄位是否存在
    existing_fields = [f.name for f in arcpy.ListFields(circle_polygon)]
    if field_name not in existing_fields:
        arcpy.management.AddField(circle_polygon, field_name, "TEXT", field_length=5)

    # 初始化為 "N"
    arcpy.management.CalculateField(circle_polygon, field_name, "'N'", "PYTHON3")

    # 建立暫存圖層進行選取
    temp_lyr = "temp_circle_lyr"
    arcpy.management.MakeFeatureLayer(circle_polygon, temp_lyr)

    # 執行按位置選取 (相交)
    arcpy.management.SelectLayerByLocation(
        in_layer=temp_lyr,
        overlap_type="INTERSECT",
        select_features=target_polygon,
        selection_type="NEW_SELECTION"
    )

    # 更新選中項為 "Y"
    arcpy.management.CalculateField(temp_lyr, field_name, "'Y'", "PYTHON3")
    
    # 清理暫存圖層
    arcpy.management.Delete(temp_lyr)


    # --- 後半段：統計與回報 ---
    
    print("-" * 30)
    print("統計回報：")

    # 使用 SearchCursor 統計數值
    counts = {"Y": 0, "N": 0}
    with arcpy.da.SearchCursor(circle_polygon, [field_name]) as cursor:
        for row in cursor:
            val = row[0]
            if val in counts:
                counts[val] += 1
    
    total = counts["Y"] + counts["N"]
    
    if total > 0:
        percent_y = (counts["Y"] / total) * 100
        percent_n = (counts["N"] / total) * 100
        
        print(f"總圖徵數量: {total}")
        print(f"相疊 (Y): {counts['Y']} 個 (佔 **{percent_y:.2f}%**)")
        print(f"未相疊 (N): {counts['N']} 個 (佔 **{percent_n:.2f}%**)")
    else:
        print("圖層中沒有任何資料可供統計。")
        
    print("-" * 30)

except Exception as e:
    print(f"發生錯誤: {e}")

------------------------------
統計回報：
總圖徵數量: 600
相疊 (Y): 206 個 (佔 **34.33%**)
未相疊 (N): 394 個 (佔 **65.67%**)
------------------------------
